# DDPG + SLM Sentiment Portfolio Workflow

This notebook runs the experimental SLM extension.

The offline model is still a DDPG model. The SLM signal is added during online evaluation.


## 0. Import shared workflow API

- Keep the notebook readable.
- Keep reusable code in `src/finance_rl_slm/`.
- Use the same output folders as the pure DDPG notebook.


In [11]:
from __future__ import annotations

import sys
from dataclasses import replace
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "main":
    PROJECT_ROOT = PROJECT_ROOT.parent

for path in (PROJECT_ROOT / "src", PROJECT_ROOT, PROJECT_ROOT / "src" / "FinRL"):
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

from finance_rl_slm.config import DEFAULT_CONFIG
from finance_rl_slm.workflow import (
    build_daily_sentiment,
    build_sentiment_inputs,
    load_online_price_data,
    load_price_data,
    plot_normalized_prices,
    print_runtime_context,
    result_picture_path,
    result_profile_path,
    run_slm_online,
    split_price_data,
)
from finance_rl_slm.evaluation import run_online_evaluation
from finance_rl_slm.training import train_offline_model


## 1. Configure the experiment

- Online test window: `2026-01-01` to `2026-06-21`.
- Pipeline name: `ddpg_slm`.
- News sample: `news_max_items = 3` per ticker for a practical local Granite run.
- SLM sentiment is treated as an experimental extra feature.


In [7]:
config = replace(
    DEFAULT_CONFIG,
    online_start="2026-01-01",
    online_end="2026-06-21",
    news_max_items=3,
)
print_runtime_context(config)


PROJECT_ROOT: /home/tonywang/114-2/PY_Finance_RL_Project
FINRL_ROOT: /home/tonywang/114-2/PY_Finance_RL_Project/src/FinRL
RSS_URLS: {'IBM': 'http://finance.yahoo.com/rss/headline?s=IBM', 'NVDA': 'http://finance.yahoo.com/rss/headline?s=NVDA', 'GM': 'http://finance.yahoo.com/rss/headline?s=GM', 'BLK': 'http://finance.yahoo.com/rss/headline?s=BLK', 'COST': 'http://finance.yahoo.com/rss/headline?s=COST'}


## 2. Run online evaluation with SLM sentiment

- This is the default safe path.
- It loads the existing `ddpg_portfolio_offline.zip` model.
- It builds weekly sentiment from a small RSS sample.
- It does not retrain the model.


In [8]:
ddpg_slm_logs = run_slm_online(config)
ddpg_slm_logs.head()


PROJECT_ROOT: /home/tonywang/114-2/PY_Finance_RL_Project
FINRL_ROOT: /home/tonywang/114-2/PY_Finance_RL_Project/src/FinRL
RSS_URLS: {'IBM': 'http://finance.yahoo.com/rss/headline?s=IBM', 'NVDA': 'http://finance.yahoo.com/rss/headline?s=NVDA', 'GM': 'http://finance.yahoo.com/rss/headline?s=GM', 'BLK': 'http://finance.yahoo.com/rss/headline?s=BLK', 'COST': 'http://finance.yahoo.com/rss/headline?s=COST'}
RAW OUTPUT: '{"label": "mixed", "confidence": 0.72}'
RAW OUTPUT: '{"label": "positive", "confidence": 0.88}'
RAW OUTPUT: '{"label": "positive", "confidence": 0.85}'
RAW OUTPUT: '{"label": "positive", "confidence": 0.88}'
RAW OUTPUT: '{"label": "positive", "confidence": 0.92}'
RAW OUTPUT: '{"label": "mixed", "confidence": 0.62}'
RAW OUTPUT: '{"label": "mixed", "confidence": 0.72}'
RAW OUTPUT: '{"label": "positive", "confidence": 0.78}'
RAW OUTPUT: '{"label": "positive", "confidence": 0.78}'
RAW OUTPUT: '{"label": "positive", "confidence": 0.78}'
RAW OUTPUT: '{"label": "positive", "confiden

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

RAW OUTPUT: '{"label": "positive", "confidence": 0.78}'
df_all_news head:
  ticker           published  \
0    IBM 2026-06-23 08:45:28   
1    IBM 2026-06-23 07:11:12   
2    IBM 2026-06-22 23:52:51   
3   NVDA 2026-06-23 09:08:00   
4   NVDA 2026-06-23 09:00:00   

                                                text sent_label  sent_conf  \
0  Is IBM Stock A Buy At $265?\nAfter a decade of...      mixed       0.72   
1  IBM (IBM) Partners With OpenAI To Bring AI Cyb...   positive       0.88   
2  Trump Provides Fresh Boost To Quantum Computin...   positive       0.85   
3  A Once-in-a-Decade Investment: This AI Stock C...   positive       0.88   
4  DDN Unveils Next-Generation AI & HPC Data Inte...   positive       0.92   

   sent_score_raw       week  
0            0.00 2026-06-23  
1            0.88 2026-06-23  
2            0.85 2026-06-16  
3            0.88 2026-06-23  
4            0.92 2026-06-23  
weekly_mkt range: 2026-06-16 00:00:00 -> 2026-06-23 00:00:00
Shape of DataFram

saved online profile: /home/tonywang/114-2/PY_Finance_RL_Project/addenda/result_profile_comparse/ddpg_slm_online_profile_2026-01-01_2026-06-21.csv


,wealth,reward,drawdown,action,sentiment,daily_return
time,,,,,,
2026-01-05,1.015932,0.015932,0.000000,"[0.39889005, 0.5643185, 0.55057245, 0.14716884...",0.0,NaN
2026-01-06,1.019479,0.003491,0.000000,"[0.39043885, 0.5600984, 0.5516676, 0.14397913,...",0.0,0.003491
2026-01-07,1.013615,-0.011504,0.005752,"[0.40459663, 0.5667938, 0.5526515, 0.1499168, ...",0.0,-0.005752
2026-01-08,1.031294,0.017442,0.000000,"[0.40726328, 0.56594276, 0.54871535, 0.1495832...",0.0,0.017442
2026-01-09,1.027334,-0.007680,0.003840,"[0.39596736, 0.5638051, 0.54590505, 0.14502138...",0.0,-0.003840


## 3. Optional offline training

- Keep this disabled during normal notebook execution.
- Turn it on only when you intentionally want to retrain the model.
- SLM is not used during offline training.


In [ ]:
# Control The DDPG retrain
RUN_OPTIONAL_TRAINING = False

if RUN_OPTIONAL_TRAINING:
    price_df = load_price_data(config)
    plot_normalized_prices(price_df, config)

    splits = split_price_data(price_df, config)
    model = train_offline_model(splits.train, splits.valid, config)
else:
    print("Optional training skipped. Set RUN_OPTIONAL_TRAINING = True to retrain.")


Optional training skipped. Set RUN_OPTIONAL_TRAINING = True to retrain.


## 4. Inspect online evaluation output

- `ddpg_slm_logs` was created by the default online cell.
- Plots and profile CSV files are saved under `addenda/`.


In [10]:
ddpg_slm_logs.head()


,wealth,reward,drawdown,action,sentiment,daily_return
time,,,,,,
2026-01-05,1.015932,0.015932,0.000000,"[0.39889005, 0.5643185, 0.55057245, 0.14716884...",0.0,NaN
2026-01-06,1.019479,0.003491,0.000000,"[0.39043885, 0.5600984, 0.5516676, 0.14397913,...",0.0,0.003491
2026-01-07,1.013615,-0.011504,0.005752,"[0.40459663, 0.5667938, 0.5526515, 0.1499168, ...",0.0,-0.005752
2026-01-08,1.031294,0.017442,0.000000,"[0.40726328, 0.56594276, 0.54871535, 0.1495832...",0.0,0.017442
2026-01-09,1.027334,-0.007680,0.003840,"[0.39596736, 0.5638051, 0.54590505, 0.14502138...",0.0,-0.003840


## 5. Output notes

- Figure files:
  - `addenda/result_picture/online_reward_ddpg_slm_2026-01-01_2026-06-21.png`
  - `addenda/result_picture/online_wealth_ddpg_slm_2026-01-01_2026-06-21.png`
  - `addenda/result_picture/online_daily_return_ddpg_slm_2026-01-01_2026-06-21.png`

- Profile file:
  - `addenda/result_profile_comparse/ddpg_slm_online_profile_2026-01-01_2026-06-21.csv`

- Compare it with the pure DDPG profile using `src/tool/compare_ddpg_profiles.py`.


## Workflow Notes - DDPG + SLM

- The SLM signal is a slow sentiment feature.

- It is not a tick-level trading signal.

- The DDPG model is trained without SLM first.

- During online evaluation, the environment reads:
  - the current trading date,
  - the daily sentiment value mapped from weekly news,
  - the same price-based technical state.

- The last observation slot stores the clipped sentiment score in `[-1, 1]`.

- This pipeline is experimental.

- The key question is simple:
  - Does a coarse sentiment feature improve the long-horizon behavior?
  - Or does it add noise to the investment policy?
